In [1]:
# Cell 1: Setup
!pip install sentence-transformers -q

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Load data
dataset_path = '/kaggle/input/dataset/split'

train_df = pd.read_csv(f'{dataset_path}/train.csv')
test_df  = pd.read_csv(f'{dataset_path}/test.csv')
val_df   = pd.read_csv(f'{dataset_path}/validation.csv')

# All splits combined
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print("Shape:", all_df.shape)
print("Columns:", all_df.columns.tolist())
print("\nLabel distribution:")
print(all_df['label'].value_counts().sort_index())
print("\nSource distribution:")
print(all_df['source'].value_counts())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.9 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [2]:
# Cell 2: Perfect Pair Matching

def normalize_val(val):
    """NaN, 'nan', 'None', '' → All empty string"""
    if pd.isna(val):
        return ""
    v = str(val).strip().lower()
    if v in ['nan', 'none', 'null', '']:
        return ""
    return v

# সব available metadata columns
match_cols = ['speaker', 'subject', 'context', 
              'speaker_job_title', 'party_affiliation', 'state_info']
available_cols = [c for c in match_cols if c in all_df.columns]
print(f"Matching columns: {available_cols}")

# Normalized key
def make_key(df, cols):
    return df[cols].apply(
        lambda col: col.map(normalize_val)
    ).agg('|'.join, axis=1)

all_df['match_key'] = make_key(all_df, available_cols)

# Separate human and AI
human_df = all_df[all_df['source'] == 'Human'].copy()
ai_df    = all_df[all_df['source'] != 'Human'].copy()

generators = ['gpt-4o-mini', 'deepseek-v3', 'claude-3-haiku', 'gemini-1.5-flash']

# Human indexed by label
human_real = human_df[human_df['label'] == 0].set_index('match_key')
human_fake = human_df[human_df['label'] == 1].set_index('match_key')

print(f"\nHuman Real samples: {len(human_real)}")
print(f"Human Fake samples: {len(human_fake)}")

# Check unique keys
h_real_unique = set(
    human_real.index.value_counts()[
        human_real.index.value_counts() == 1
    ].index
)
h_fake_unique = set(
    human_fake.index.value_counts()[
        human_fake.index.value_counts() == 1
    ].index
)

print(f"Unique Human Real keys: {len(h_real_unique)}")
print(f"Unique Human Fake keys: {len(h_fake_unique)}")

# Per generator matching check
print(f"\n{'Generator':<22} {'Matched':>8} {'Total AI':>10} {'Match%':>8}")
print("-"*52)

for gen in generators:
    gen_df   = ai_df[ai_df['source'] == gen]
    gen_real = gen_df[gen_df['label'] == 2]
    gen_fake = gen_df[gen_df['label'] == 3]
    
    a_real_unique = set(
        gen_real['match_key'].value_counts()[
            gen_real['match_key'].value_counts() == 1
        ].index
    )
    a_fake_unique = set(
        gen_fake['match_key'].value_counts()[
            gen_fake['match_key'].value_counts() == 1
        ].index
    )
    
    matched_real = h_real_unique & a_real_unique
    matched_fake = h_fake_unique & a_fake_unique
    total_matched = len(matched_real) + len(matched_fake)
    total_ai      = len(gen_df)
    pct           = total_matched / total_ai * 100
    
    print(f"{gen:<22} {total_matched:>8} {total_ai:>10} {pct:>7.1f}%")

# Sample pair দেখাও
print("\n--- Sample Pairs ---")
for gen in generators:
    gen_df   = ai_df[ai_df['source'] == gen]
    gen_real = gen_df[gen_df['label'] == 2].set_index('match_key')
    
    a_real_unique = set(
        gen_real.index.value_counts()[
            gen_real.index.value_counts() == 1
        ].index
    )
    matched = list(h_real_unique & a_real_unique)
    
    if matched:
        key = matched[0]
        h = human_real.loc[key, 'statement']
        a = gen_real.loc[key, 'statement']
        if isinstance(h, pd.Series): h = h.iloc[0]
        if isinstance(a, pd.Series): a = a.iloc[0]
        print(f"\n{gen}:")
        print(f"  Human: {str(h)[:100]}")
        print(f"  AI:    {str(a)[:100]}")

Matching columns: ['speaker', 'subject', 'context', 'speaker_job_title', 'party_affiliation', 'state_info']

Human Real samples: 4503
Human Fake samples: 5643
Unique Human Real keys: 4427
Unique Human Fake keys: 5536

Generator               Matched   Total AI   Match%
----------------------------------------------------
gpt-4o-mini                2930       2994    97.9%
deepseek-v3                2916       2991    97.5%
claude-3-haiku             2116       2146    98.6%
gemini-1.5-flash           1950       1999    97.5%

--- Sample Pairs ---

gpt-4o-mini:
  Human: The race will tighten, just because that's what happens at the end of campaigns. They always have.
  AI:    The competition is bound to become more intense as we near the end of the campaign, as has always be

deepseek-v3:
  Human: In Rhode Island, 28 percent of adults released from state prisons are re-incarcerated within a year.
  AI:    In Rhode Island, 28% of adults released from state prison end up back behind bars 

In [5]:
# Cell 3: Semantic Similarity Analysis

print("Loading SBERT...")
sbert = SentenceTransformer('all-MiniLM-L6-v2')
print("✓ Loaded!\n")

def get_similarity(texts1, texts2):
    e1 = sbert.encode(texts1, batch_size=64, show_progress_bar=False)
    e2 = sbert.encode(texts2, batch_size=64, show_progress_bar=False)
    return np.array([
        cosine_similarity([a], [b])[0][0] 
        for a, b in zip(e1, e2)
    ])

# Length ratio per generator (threshold জন্য)
human_avg_len = human_df['statement'].str.len().mean()
length_ratios = {}
for gen in generators:
    ai_avg = ai_df[ai_df['source'] == gen]['statement'].str.len().mean()
    length_ratios[gen] = ai_avg / human_avg_len

print("Length Ratios:")
for gen, ratio in length_ratios.items():
    t = 0.70
    print(f"  {gen:<22}: {ratio:.2f}x → threshold={t}")

# Main analysis
print("\n" + "="*70)
results      = {}
all_sims_all = []

for gen in generators:
    print(f"\nProcessing: {gen}")
    
    gen_df   = ai_df[ai_df['source'] == gen]
    gen_real = gen_df[gen_df['label'] == 2].set_index('match_key')
    gen_fake = gen_df[gen_df['label'] == 3].set_index('match_key')
    
    # Unique keys
    a_real_u = set(gen_real.index.value_counts()[
                   gen_real.index.value_counts()==1].index)
    a_fake_u = set(gen_fake.index.value_counts()[
                   gen_fake.index.value_counts()==1].index)
    
    matched_real = list(h_real_unique & a_real_u)[:200]
    matched_fake = list(h_fake_unique & a_fake_u)[:200]
    
    # Collect text pairs
    h_real_txt, a_real_txt = [], []
    for key in matched_real:
        try:
            h = human_real.loc[key, 'statement']
            a = gen_real.loc[key, 'statement']
            if isinstance(h, pd.Series): h = h.iloc[0]
            if isinstance(a, pd.Series): a = a.iloc[0]
            h_real_txt.append(str(h))
            a_real_txt.append(str(a))
        except: continue

    h_fake_txt, a_fake_txt = [], []
    for key in matched_fake:
        try:
            h = human_fake.loc[key, 'statement']
            a = gen_fake.loc[key, 'statement']
            if isinstance(h, pd.Series): h = h.iloc[0]
            if isinstance(a, pd.Series): a = a.iloc[0]
            h_fake_txt.append(str(h))
            a_fake_txt.append(str(a))
        except: continue

    print(f"  Real pairs: {len(h_real_txt)} | Fake pairs: {len(h_fake_txt)}")

    sim_real = get_similarity(h_real_txt, a_real_txt) \
               if h_real_txt else np.array([])
    sim_fake = get_similarity(h_fake_txt, a_fake_txt) \
               if h_fake_txt else np.array([])
    all_sim  = np.concatenate([sim_real, sim_fake])
    all_sims_all.append(all_sim)

    ratio     = length_ratios[gen]
    threshold = 0.70
    noise_pct = float(np.mean(all_sim < threshold) * 100)

    results[gen] = {
        'real_mean':  float(np.mean(sim_real)) if len(sim_real) > 0 else 0,
        'real_std':   float(np.std(sim_real))  if len(sim_real) > 0 else 0,
        'fake_mean':  float(np.mean(sim_fake)) if len(sim_fake) > 0 else 0,
        'fake_std':   float(np.std(sim_fake))  if len(sim_fake) > 0 else 0,
        'overall':    float(np.mean(all_sim)),
        'noise_pct':  noise_pct,
        'threshold':  threshold,
        'ratio':      ratio,
        'n_real':     len(sim_real),
        'n_fake':     len(sim_fake),
        'n_total':    len(all_sim),
        # Raw for spot check
        'h_real': h_real_txt,
        'a_real': a_real_txt,
        'sim_real_raw': sim_real
    }

    print(f"  Real sim: {results[gen]['real_mean']:.4f} ± {results[gen]['real_std']:.4f}")
    print(f"  Fake sim: {results[gen]['fake_mean']:.4f} ± {results[gen]['fake_std']:.4f}")
    print(f"  Noise (threshold={threshold}): {noise_pct:.2f}%")

# Summary Table
print("\n" + "="*80)
print("SEMANTIC EQUIVALENCE AUDIT — FINAL RESULTS")
print("="*80)
print(f"{'Generator':<22} {'Real Sim':>16} {'Fake Sim':>16} "
      f"{'Overall':>9} {'Noise%':>8} {'N':>6}")
print("-"*80)

for gen, res in results.items():
    r = f"{res['real_mean']:.4f}±{res['real_std']:.4f}"
    f_ = f"{res['fake_mean']:.4f}±{res['fake_std']:.4f}"
    print(f"{gen:<22} {r:>16} {f_:>16} "
          f"{res['overall']:>9.4f} {res['noise_pct']:>7.2f}% "
          f"{res['n_total']:>6}")

all_flat  = np.concatenate(all_sims_all)
avg_noise = np.mean([r['noise_pct'] for r in results.values()])
print("-"*80)
print(f"{'Overall':<22} {'':>16} {'':>16} "
      f"{np.mean(all_flat):>9.4f} {avg_noise:>7.2f}%")
print("="*80)

Loading SBERT...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Loaded!

Length Ratios:
  gpt-4o-mini           : 1.13x → threshold=0.7
  deepseek-v3           : 1.09x → threshold=0.7
  claude-3-haiku        : 2.10x → threshold=0.7
  gemini-1.5-flash      : 1.11x → threshold=0.7


Processing: gpt-4o-mini
  Real pairs: 200 | Fake pairs: 200
  Real sim: 0.8630 ± 0.1142
  Fake sim: 0.8501 ± 0.0812
  Noise (threshold=0.7): 6.00%

Processing: deepseek-v3
  Real pairs: 200 | Fake pairs: 200
  Real sim: 0.9050 ± 0.0587
  Fake sim: 0.8378 ± 0.0785
  Noise (threshold=0.7): 2.25%

Processing: claude-3-haiku
  Real pairs: 200 | Fake pairs: 200
  Real sim: 0.7973 ± 0.1180
  Fake sim: 0.6584 ± 0.1274
  Noise (threshold=0.7): 35.00%

Processing: gemini-1.5-flash
  Real pairs: 200 | Fake pairs: 200
  Real sim: 0.8255 ± 0.0909
  Fake sim: 0.7691 ± 0.1132
  Noise (threshold=0.7): 18.25%

SEMANTIC EQUIVALENCE AUDIT — FINAL RESULTS
Generator                      Real Sim         Fake Sim   Overall   Noise%      N
----------------------------------------------------

In [6]:
# Cell 4: Spot Check — 3 Lowest Pairs Per Generator

print("SPOT CHECK — 3 Lowest Similarity Pairs Per Generator")

for gen in generators:
    print(f"\n{'='*65}")
    print(f"Generator: {gen}  (threshold={results[gen]['threshold']})")
    print('='*65)
    
    h_txts  = results[gen]['h_real']
    a_txts  = results[gen]['a_real']
    sims    = results[gen]['sim_real_raw']
    
    if len(sims) == 0:
        print("  No pairs available")
        continue
    
    sorted_idx = np.argsort(sims)
    
    for i, idx in enumerate(sorted_idx[:3]):
        print(f"\n  Pair {i+1} | Similarity: {sims[idx]:.4f}")
        print(f"  Human: {h_txts[idx][:120]}")
        print(f"  AI:    {a_txts[idx][:120]}")

SPOT CHECK — 3 Lowest Similarity Pairs Per Generator

Generator: gpt-4o-mini  (threshold=0.7)

  Pair 1 | Similarity: 0.1687
  Human: The Associated Press called Charlie Crist's attacks "over the top," "out of context," and "not true."
  AI:    excessive,

  Pair 2 | Similarity: 0.2812
  Human: Says the Obama administration never responded to his 2012 letter flagging the uptick in unaccompanied children crossing 
  AI:    Mexico border.

  Pair 3 | Similarity: 0.3221
  Human: While in the Illinois Senate, Barack Obama passed "tax cuts for hard-working families."
  AI:    working families.

Generator: deepseek-v3  (threshold=0.7)

  Pair 1 | Similarity: 0.7007
  Human: Under Hosni Mubaraks rule, Egypt received more American dollars than any country besides Israel.
  AI:    Under Hosni Mubarak, Egypt was the second-largest recipient of U.S. aid after Israel.*)

  Pair 2 | Similarity: 0.7053
  Human: The House has never failed to pass a budget in the modern era.
  AI:    In the modern era